# ATLAS Wind Downscaling for Climate Scenarios

This notebook performs statistical downscaling of climate scenario wind components using ERA5Land topographic predictors and GLO90 high resolution topography.

It replaces the previous separate notebooks for **islands** and **continental** domains with one single workflow.

The methodology is shared by both cases. The only operational difference is:

- `AREA_MODE = "islands"` processes one bbox directly.
- `AREA_MODE = "continental"` trains one model on the full bbox, then splits only the high resolution prediction into latitude bands to reduce memory use.

The bbox format used throughout the notebook is:

`[lat_max, lon_min, lat_min, lon_max]`

## Methodology Description

This notebook applies a Machine Learning based statistical downscaling approach to generate high-resolution climate projections from coarse-resolution CMIP6 climate model outputs. The method relies on a **Multi-Layer Perceptron (MLP)** neural network trained to learn the relationship between large-scale climate model variables and local terrain characteristics.

The downscaling procedure uses a set of predictors derived from both the CMIP6 climate simulations and the Copernicus GLO90 Digital Elevation Model. The target variable depends on the application and may include temperature, precipitation, solar radiation, or wind-related variables. For each target variable, the corresponding CMIP6 predictor is used together with terrain descriptors that represent the influence of local topography on climate conditions.

During the training phase, the MLP is calibrated using the coarse-resolution CMIP6 variable together with the associated topographic predictors. The trained model is subsequently applied to the high-resolution GLO90 grid, allowing the generation of climate projections at a much finer spatial resolution.

Topographic information derived from the Digital Elevation Model is a key component of the methodology. Variables such as elevation, slope, and terrain aspect provide detailed spatial descriptors that are not explicitly resolved by global climate models, enabling the neural network to reproduce local-scale spatial variability driven by terrain characteristics.

This approach preserves the large-scale climate signal provided by the CMIP6 simulations while enhancing its spatial detail, producing high-resolution climate projections suitable for local impact assessments, climate adaptation planning, and sector-specific analyses.

In the current implementation, the predictor set consists of elevation, slope, and transformed aspect variables (sine and cosine of aspect), which allow the neural network to capture terrain-driven variations in local climate conditions while avoiding discontinuities associated with circular angular measurements.

## 1. Import libraries

Run this cell first. The notebook was designed for the same Python environment used by the other ATLAS wind notebooks.


In [1]:
from pathlib import Path
import gc

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

import numpy as np
import xarray as xr
import pandas as pd
import rioxarray

try:
    import geopandas as gpd
    import matplotlib.pyplot as plt
except ImportError:
    gpd = None
    plt = None


## 2. User configuration

Edit only this cell before running the notebook.

Main choices:

- `comp`: wind component to downscale. Use `"u"` or `"v"`.
- `exp`: climate experiment, for example `"historical"`, `"ssp245"`, `"ssp370"` or `"ssp585"`.
- `model`: climate model name.
- `AREA_MODE`: use `"islands"` for a small domain, `"continental"` for a large continental area.
- `AREA_NAME`: by default it follows `AREA_MODE`. Change it only if you add a named bbox in `BBOXES`.

The notebook expects that the preprocessing notebook has already created a file such as:

`uas_2015-01_2100-12_processed.nc`

or

`vas_2015-01_2100-12_processed.nc`

depending on `comp`.


In [2]:
# =============================================================================
# USER SETTINGS
# =============================================================================

# Country or region label used in input and output file names.
country = "chile"

# Month to downscale.
# Use an integer from 1 to 12.
month = 1

# Wind component.
# Use "u" for the eastward wind component.
# Use "v" for the northward wind component.
comp = "v"

# Climate scenario settings.
# Examples: "historical", "ssp245", "ssp370", "ssp585".
exp = "ssp370"
model = "CNRM-ESM2-1"

# Period to extract from the preprocessed scenario file.
# These years are used in the output file name.
start_period = "2020"
end_period = "2050"

# Input scenario file period.
# These dates must match the name of the preprocessed scenario file.
if exp == "historical":
    start_date = "1985-01"
    end_date = "2014-12"
else:
    start_date = "2015-01"
    end_date = "2100-12"

# Choose the processing strategy.
# Use "islands" for island boxes or small domains.
# Use "continental" for large continental domains that should be processed in chunks.
AREA_MODE = "islands"

# Bounding boxes.
# Format: [lat_max, lon_min, lat_min, lon_max].
# For Chile, latitudes are negative. lat_max is the northern boundary.
BBOXES = {
    "continental": [-17.3, -76.2, -56.7, -66.2],
    "islands": [-26.0, -109.8, -34.8, -77.8],
}

# AREA_NAME follows AREA_MODE by default.
# Change it only if you add another named bbox to BBOXES.
AREA_NAME = AREA_MODE
AREA_BBOX = BBOXES[AREA_NAME]

# Continental split settings.
# The model is trained once on the full continental bbox.
# Only the high resolution GLO90 prediction is split into latitude bands.
CONTINENTAL_N_SPLITS = 2
CONTINENTAL_BUFFER_DEG = 1.0

# Generic project paths.
# Keep these paths relative so that the notebook can run on different machines.
PROJECT_DIR = Path("../data")
DEM_DIR = Path("../DEMdata") / country

# Folder containing the scenario component produced by the preprocessing notebook.
SCENARIO_COMPONENT_DIR = PROJECT_DIR / "processed" / f"10m_wind_{comp}_component" / country / model / exp

# Folder where final downscaled products are written.
OUTPUT_DIR = PROJECT_DIR / "downscaled_data" / country / model / exp 

# Folder where continental chunks are written.
SUBAREA_OUTPUT_DIR = OUTPUT_DIR / "sub_areas"

# Input file names.
# Change these only if your previous notebooks produced different names.
SCENARIO_COMPONENT_FILE = SCENARIO_COMPONENT_DIR / f"{comp}as_{start_date}_{end_date}_{country}_{exp}_{model}_processed.nc"

ERA5_OROGRAPHY_FILE = DEM_DIR / f"era5land_orography_{country}.nc"
ERA5_ASPECT_FILE = DEM_DIR / f"era5land_aspect_{country}.nc"
GLO90_OROGRAPHY_FILE = DEM_DIR / f"glo90_orography_{country}_{AREA_NAME}.nc"
GLO90_ASPECT_FILE = DEM_DIR / f"glo90_aspect_{country}_{AREA_NAME}.nc"

# Optional shapefile for visual checks.
# Leave as None if you do not need plotting.
SHAPEFILE_PATH = None

# Model settings.
MLP_RANDOM_STATE = 1
MLP_MAX_ITER = 1000


## 3. Create output folders and check inputs

This cell creates the output folders and checks whether the required input files exist.

If an error appears here, update the paths in the configuration cell before continuing.


In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBAREA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    SCENARIO_COMPONENT_FILE,
    ERA5_OROGRAPHY_FILE,
    ERA5_ASPECT_FILE,
    GLO90_OROGRAPHY_FILE,
    GLO90_ASPECT_FILE,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    print("Missing input files:")
    for path in missing_files:
        print(f"  {path}")
    raise FileNotFoundError("Please update the paths in the configuration cell before continuing.")

print("All required input files were found.")
print(f"Experiment: {exp}")
print(f"Model: {model}")
print(f"Component: {comp}as")
print(f"Selected period: {start_period} to {end_period}")
print(f"Area mode: {AREA_MODE}")
print(f"Area name: {AREA_NAME}")
print(f"Area bbox: {AREA_BBOX}")


All required input files were found.
Experiment: ssp370
Model: CNRM-ESM2-1
Component: vas
Selected period: 2020 to 2050
Area mode: islands
Area name: islands
Area bbox: [-26.0, -109.8, -34.8, -77.8]


## 4. Helper functions

These functions are shared by islands and continental workflows.


In [4]:
def drop_spatial_ref(obj):
    """Drop common CRS helper variables when they are present."""
    return obj.drop_vars(["spatial_ref", "crs"], errors="ignore")


def save_xarray_netcdf_fast(ds, output_path):
    """Save an xarray Dataset as compressed NetCDF."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    encoding = {
        var: {"zlib": True, "complevel": 1}
        for var in ds.data_vars
    }

    ds.to_netcdf(
        output_path,
        engine="netcdf4",
        encoding=encoding,
    )


def validate_bbox(bbox):
    """Validate bbox format: [lat_max, lon_min, lat_min, lon_max]."""
    if len(bbox) != 4:
        raise ValueError("BBOX must have four values: [lat_max, lon_min, lat_min, lon_max].")

    lat_max, lon_min, lat_min, lon_max = bbox

    if lat_max <= lat_min:
        raise ValueError("lat_max must be greater than lat_min. Remember that Chile latitudes are negative.")
    if lon_min >= lon_max:
        raise ValueError("lon_min must be smaller than lon_max.")


def split_bbox_by_latitude(bbox, n_splits=2, buffer_deg=0.0):
    """Split a bbox into latitude bands.

    The returned boxes overlap by buffer_deg. This helps avoid edge artefacts during the final merge.
    """
    validate_bbox(bbox)
    lat_max, lon_min, lat_min, lon_max = bbox

    if n_splits < 1:
        raise ValueError("n_splits must be at least 1.")

    if n_splits == 1:
        return {"full": bbox}

    edges = np.linspace(lat_max, lat_min, n_splits + 1)
    subareas = {}

    for idx in range(n_splits):
        north_edge = edges[idx]
        south_edge = edges[idx + 1]

        sub_lat_max = min(lat_max, north_edge + buffer_deg if idx > 0 else north_edge)
        sub_lat_min = max(lat_min, south_edge - buffer_deg if idx < n_splits - 1 else south_edge)

        if n_splits == 2:
            label = "north" if idx == 0 else "south"
        else:
            label = f"band_{idx + 1:02d}"

        subareas[label] = [float(sub_lat_max), lon_min, float(sub_lat_min), lon_max]

    return subareas


def select_bbox(ds, bbox):
    """Select a bbox from an xarray object using latitude and longitude.

    This helper works whether latitude is ascending or descending.
    """
    validate_bbox(bbox)
    lat_max, lon_min, lat_min, lon_max = bbox

    lat = ds["latitude"]
    lon = ds["longitude"]

    lat_slice = slice(lat_max, lat_min) if lat[0] > lat[-1] else slice(lat_min, lat_max)
    lon_slice = slice(lon_min, lon_max) if lon[0] < lon[-1] else slice(lon_max, lon_min)

    selected = ds.sel(
        longitude=lon_slice,
        latitude=lat_slice,
    )

    if selected.sizes.get("latitude", 0) == 0 or selected.sizes.get("longitude", 0) == 0:
        print("Warning: selected bbox has an empty grid.")
        print(f"  bbox: {bbox}")
        print(f"  latitude range in dataset: {float(lat.min())} to {float(lat.max())}")
        print(f"  longitude range in dataset: {float(lon.min())} to {float(lon.max())}")

    return selected


def add_aspect_sin_cos(df):
    """Convert aspect degrees into sine and cosine features.

    Aspect is circular. Using sine and cosine avoids discontinuities between 359 and 0 degrees.
    """
    df = df.copy()

    if "aspect" in df.columns:
        df["aspect_sin"] = np.sin(np.deg2rad(df["aspect"]))
        df["aspect_cos"] = np.cos(np.deg2rad(df["aspect"]))
        df = df.drop(columns=["aspect"])

    return df


## 5. Machine learning functions

The same training and prediction functions are used for islands and continental areas.

The target is the scenario wind component:

- `uas` for the u component
- `vas` for the v component

The predictors are high resolution topographic variables derived from GLO90.


In [5]:
def era5scaler(xdf, features, target):
    """Scale features and target using StandardScaler."""
    featurescaler = StandardScaler()
    X_scaled = featurescaler.fit_transform(xdf[features])

    targetscaler = StandardScaler()
    y_scaled = targetscaler.fit_transform(xdf[[target]]).ravel()

    return featurescaler, targetscaler, X_scaled, y_scaled


def make_dataset_era5land(target_var, orography, aspect):
    """Create the training dataframe by merging scenario component and ERA5Land DEM features.

    The scenario component is first averaged by calendar month.
    The selected month is handled in training_ds().
    """
    target_var_monthly = target_var.groupby(target_var.time.dt.month).mean()

    xdf_merged = xr.merge([
        orography,
        aspect,
        target_var_monthly,
    ])

    merged = xdf_merged.to_dataframe().dropna().reset_index()
    return merged


def training_ds(era5l_merged, month, target):
    """Prepare the monthly training dataframe and train the MLP model."""
    era5l_merged = (
        era5l_merged
        .loc[era5l_merged.month == month, :]
        .drop("month", axis=1)
    )

    era5l_merged = add_aspect_sin_cos(era5l_merged)

    features = era5l_merged.columns.drop([
        target,
        "latitude",
        "longitude",
    ])

    featurescaler_era5, targetscaler_era5, era5l_X, era5l_y = era5scaler(
        era5l_merged,
        features,
        target,
    )

    regr = MLPRegressor(
        random_state=MLP_RANDOM_STATE,
        max_iter=MLP_MAX_ITER,
    ).fit(era5l_X, era5l_y)

    return features, featurescaler_era5, targetscaler_era5, era5l_X, era5l_y, regr


def apply_downscaling(dataset_glo90, features, template_da, comp, featurescaler_era5, targetscaler_era5, regr):
    """Apply the trained model to the GLO90 dataframe and map predictions back to the grid."""
    df = dataset_glo90.copy()
    df = add_aspect_sin_cos(df)

    if df.empty:
        return xr.Dataset({f"{comp}as_downscaled": xr.full_like(template_da, np.nan, dtype=float)})

    missing_features = [feature for feature in features if feature not in df.columns]
    if missing_features:
        raise ValueError(f"The GLO90 dataframe is missing these model features: {missing_features}")

    glo90_X = featurescaler_era5.transform(df[features])
    downscaled_std = regr.predict(glo90_X)
    downscaled = targetscaler_era5.inverse_transform(
        downscaled_std.reshape(-1, 1)
    ).ravel()

    out = xr.full_like(template_da, np.nan, dtype=float)
    out_values = out.values.copy()

    lat_idx = template_da.get_index("latitude").get_indexer(df["latitude"].to_numpy())
    lon_idx = template_da.get_index("longitude").get_indexer(df["longitude"].to_numpy())

    missing = (lat_idx < 0) | (lon_idx < 0)
    if missing.any():
        raise ValueError(
            f"Some dataframe coordinates were not found in the template grid: {missing.sum()} missing points. "
            "Check latitude and longitude precision."
        )

    out_values[lat_idx, lon_idx] = downscaled
    out.values[:] = out_values

    return xr.Dataset({
        f"{comp}as_downscaled": out,
    })


## 6. Data loading functions

This section loads:

- the preprocessed scenario wind component;
- ERA5Land orography and aspect for model training;
- GLO90 orography and aspect for high resolution prediction.

The scenario component is interpolated to the ERA5Land static grid before training.


In [6]:
def load_input_datasets():
    """Load all input datasets and align them to the grids used by the workflow."""
    print("Opening and preparing scenario wind component...")

    component = (
        xr.open_mfdataset(SCENARIO_COMPONENT_FILE)
        .sel(latitude=slice(None, None, -1))
        .sel(time=slice(start_period, end_period))
    )
    component = drop_spatial_ref(component.rio.write_crs("EPSG:4326"))

    print("Opening ERA5Land static predictors...")
    orography = xr.open_dataset(ERA5_OROGRAPHY_FILE)
    orography = drop_spatial_ref(orography.rio.write_crs("EPSG:4326"))

    aspect = xr.open_dataset(ERA5_ASPECT_FILE)
    aspect = drop_spatial_ref(aspect.rio.write_crs("EPSG:4326"))

    print("Interpolating scenario component and aspect to the ERA5Land orography grid...")
    component_interp = component.interp(
        latitude=orography.latitude.values,
        longitude=orography.longitude.values,
        method="slinear",
    )

    aspect_interp = aspect.interp(
        latitude=orography.latitude.values,
        longitude=orography.longitude.values,
        method="nearest",
    )

    print("Opening GLO90 high resolution predictors...")
    orography_glo90 = xr.open_dataset(GLO90_OROGRAPHY_FILE)
    orography_glo90 = drop_spatial_ref(orography_glo90.rio.write_crs("EPSG:4326"))

    aspect_glo90 = xr.open_dataset(GLO90_ASPECT_FILE)
    aspect_glo90 = drop_spatial_ref(aspect_glo90.rio.write_crs("EPSG:4326"))

    print("Interpolating GLO90 aspect to the GLO90 orography grid...")
    aspect_glo90_interp = aspect_glo90.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="nearest",
    )

    return {
        "component_interp": component_interp,
        "orography": orography,
        "aspect_interp": aspect_interp,
        "orography_glo90": orography_glo90,
        "aspect_glo90_interp": aspect_glo90_interp,
    }


## 7. Downscaling workflow functions

The functions below keep the methodology common between islands and continental processing.

For `AREA_MODE = "continental"`, the model is trained once on the full continental bbox. Only the prediction on the GLO90 grid is split into bands.


In [7]:
def train_model_for_bbox(datasets, bbox):
    """Train the downscaling model on the selected bbox."""
    print("Preparing training dataset...")

    orography_area = select_bbox(datasets["orography"], bbox)
    aspect_area = select_bbox(datasets["aspect_interp"], bbox)
    component_area = select_bbox(datasets["component_interp"], bbox)

    era5l_merged = make_dataset_era5land(
        component_area,
        orography_area,
        aspect_area,
    )

    target = f"{comp}as"

    print("Training model...")
    features, featurescaler_era5, targetscaler_era5, era5l_X, era5l_y, regr = training_ds(
        era5l_merged,
        month,
        target,
    )

    print("Model features:", list(features))

    return {
        "features": features,
        "featurescaler_era5": featurescaler_era5,
        "targetscaler_era5": targetscaler_era5,
        "regr": regr,
    }


def make_glo90_dataframe_for_bbox(datasets, bbox):
    """Create the GLO90 prediction dataframe for the selected bbox."""
    orography_glo90_area = select_bbox(datasets["orography_glo90"], bbox)
    aspect_glo90_area = select_bbox(datasets["aspect_glo90_interp"], bbox)

    oro = drop_spatial_ref(orography_glo90_area["z"])
    asp = drop_spatial_ref(aspect_glo90_area["aspect"])

    oro, asp = xr.align(oro, asp, join="exact")

    valid_mask = oro.notnull() & asp.notnull()
    oro = oro.where(valid_mask)
    asp = asp.where(valid_mask)

    glo90_ds = xr.Dataset({
        "z": oro,
        "aspect": asp,
    })

    dataset_glo90 = (
        glo90_ds
        .to_dataframe()
        .dropna()
        .reset_index()
    )

    print(f"Valid GLO90 pixels: {len(dataset_glo90)}")
    return glo90_ds, dataset_glo90


def downscale_prediction_bbox(datasets, model, bbox):
    """Apply the trained model to one GLO90 bbox."""
    glo90_ds, dataset_glo90 = make_glo90_dataframe_for_bbox(datasets, bbox)

    if dataset_glo90.empty:
        print("No valid GLO90 pixels were found for this bbox. The chunk will be skipped.")
        return None

    output_ds = apply_downscaling(
        dataset_glo90,
        model["features"],
        glo90_ds["z"],
        comp,
        model["featurescaler_era5"],
        model["targetscaler_era5"],
        model["regr"],
    )

    return output_ds


def run_islands_workflow(datasets, area_name, bbox):
    """Run the workflow for islands or small domains without splitting."""
    print(f"Running islands workflow for {area_name}...")

    model = train_model_for_bbox(datasets, bbox)
    output_ds = downscale_prediction_bbox(datasets, model, bbox)

    if output_ds is None:
        raise ValueError("No valid GLO90 pixels were found for this bbox.")

    output_file = OUTPUT_DIR / (
        f"{comp}as_component_downscaled_{country}_m{month}_{area_name}_{start_period}_{end_period}.nc"
    )

    print(f"Saving: {output_file}")
    save_xarray_netcdf_fast(output_ds, output_file)

    return output_ds, output_file


def run_continental_workflow(datasets, area_name, bbox):
    """Run the continental workflow.

    The model is trained once on the full continental bbox.
    The prediction is split into latitude bands to reduce memory usage.
    The chunks are saved separately and then merged into one final output file.
    """
    print(f"Running continental workflow for {area_name}...")
    print("Training uses the full continental bbox.")

    model = train_model_for_bbox(datasets, bbox)

    subareas = split_bbox_by_latitude(
        bbox,
        n_splits=CONTINENTAL_N_SPLITS,
        buffer_deg=CONTINENTAL_BUFFER_DEG,
    )

    output_files = []

    for subarea_name, subarea_bbox in subareas.items():
        print(f"Processing continental subarea: {subarea_name}")
        print(f"Subarea bbox: {subarea_bbox}")

        output_ds = downscale_prediction_bbox(datasets, model, subarea_bbox)

        if output_ds is None:
            continue

        output_file = SUBAREA_OUTPUT_DIR / (
            f"{comp}as_component_downscaled_{country}_m{month}_{area_name}_{subarea_name}_{start_period}_{end_period}.nc"
        )

        print(f"Saving: {output_file}")
        save_xarray_netcdf_fast(output_ds, output_file)
        output_files.append(output_file)

        del output_ds
        gc.collect()

    if not output_files:
        raise ValueError(
            "No continental chunk produced valid output. Check AREA_BBOX, longitude convention and input files."
        )

    print("Merging continental subareas...")

    merged = None

    for output_file in output_files:
        print(f"Opening chunk: {output_file}")

        ds = xr.open_dataset(
            output_file,
            chunks={
                "latitude": 1000,
                "longitude": 1000,
            },
        )

        ds = drop_spatial_ref(ds.rio.write_crs("EPSG:4326"))

        if "latitude" in ds.coords:
            ds = ds.sortby("latitude")
        if "longitude" in ds.coords:
            ds = ds.sortby("longitude")

        if merged is None:
            merged = ds
        else:
            merged = merged.combine_first(ds)

    print("Sorting final grid...")

    if "latitude" in merged.coords:
        merged = merged.sortby("latitude")
    if "longitude" in merged.coords:
        merged = merged.sortby("longitude")

    final_output_file = OUTPUT_DIR / (
        f"{comp}as_component_downscaled_{country}_m{month}_{area_name}_{start_period}_{end_period}.nc"
    )

    print(f"Saving merged continental output: {final_output_file}")
    save_xarray_netcdf_fast(merged, final_output_file)

    return merged, final_output_file, output_files


## 8. Run the downscaling

This cell chooses the correct workflow based on `AREA_MODE`.

Use:

- `AREA_MODE = "islands"` for one small bbox;
- `AREA_MODE = "continental"` for a large bbox that should be split during prediction.


In [8]:
validate_bbox(AREA_BBOX)

datasets = load_input_datasets()

if AREA_MODE == "islands":
    output_ds, final_output_file = run_islands_workflow(
        datasets,
        AREA_NAME,
        AREA_BBOX,
    )
    chunk_output_files = []

elif AREA_MODE == "continental":
    output_ds, final_output_file, chunk_output_files = run_continental_workflow(
        datasets,
        AREA_NAME,
        AREA_BBOX,
    )

else:
    raise ValueError("AREA_MODE must be either 'islands' or 'continental'.")

print("Done.")
print(f"Final output: {final_output_file}")

if chunk_output_files:
    print("Chunk outputs:")
    for path in chunk_output_files:
        print(f"  {path}")


Opening and preparing scenario wind component...


/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(
ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


Opening ERA5Land static predictors...
Interpolating scenario component and aspect to the ERA5Land orography grid...
Opening GLO90 high resolution predictors...
Interpolating GLO90 aspect to the GLO90 orography grid...
Running islands workflow for islands...
Preparing training dataset...
Training model...


/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Model features: ['z', 'aspect_sin', 'aspect_cos']
Valid GLO90 pixels: 38552
Saving: ../data/downscaled_data/chile/CNRM-ESM2-1/ssp370/vas_component_downscaled_chile_m1_islands_2020_2050.nc
Done.
Final output: ../data/downscaled_data/chile/CNRM-ESM2-1/ssp370/vas_component_downscaled_chile_m1_islands_2020_2050.nc


## 9. Optional quick plot

Use this section only to visually inspect the output.

If `SHAPEFILE_PATH` is `None`, the plot will show only the raster output.


In [9]:
def plot_downscaled_output(ds, variable=None, shapefile_path=None):
    """Plot the downscaled output with an optional shapefile overlay."""
    if plt is None:
        raise ImportError("matplotlib is not available in this environment.")

    if variable is None:
        variable = f"{comp}as_downscaled"

    da = ds[variable]

    try:
        da = da.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
        da = da.rio.write_crs("EPSG:4326", inplace=False)
    except Exception:
        pass

    fig, ax = plt.subplots(figsize=(10, 7))
    da.plot(ax=ax)

    if shapefile_path is not None:
        if gpd is None:
            raise ImportError("geopandas is not available in this environment.")
        geometries = gpd.read_file(shapefile_path).to_crs("EPSG:4326")
        geometries.plot(ax=ax, edgecolor="black", facecolor="none")

    ax.set_title(variable)
    plt.show()


# Uncomment the next line to plot the result.
# plot_downscaled_output(output_ds, shapefile_path=SHAPEFILE_PATH)


## 10. Notes for operators

For large continental domains, increase `CONTINENTAL_N_SPLITS` if the notebook runs out of memory. For example, use `3` or `4` instead of `2`.

The output of each continental chunk is stored in `SUBAREA_OUTPUT_DIR`. The final merged output is stored in `OUTPUT_DIR`.

For island domains, the notebook creates only one output file.

Run the notebook once for `comp = "u"` and once for `comp = "v"` if both wind components are needed for the later conversion step.
